In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np


#Read excel files

df_stroop = pd.read_excel("Raw_Stroop.xlsx")

#Save as CSV files in the same folder

df_stroop.to_csv("Raw_Stroop.csv", index=False)


In [2]:
#Visa sidan "Variables description" i excel filen

desc_stroop = pd.read_excel("Raw_Stroop.xlsx", sheet_name = "Variables Description")
pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_rows", None)
desc_stroop

,Variable name,Variable Description
0,build,The Inquisit version used to run the task
1,computer.platform,The operational system the task was run on (win/mac/ios/android)
2,date,The date the task was executed
3,time,The time the task was executed
4,blockcode,"The name of the current block (""Stroop"" for all trials)"
5,blocknum,The number of the current block (2 for all trials)
6,trialcode,The name of the current trial:\n“blackcongruent”=congruent trial with black font\n“blackincongruent”=incongruent trial with black font\n“bluecongruent”=congruent trial with blue font\n“blueincongruent”=incongruent trial with blue font\n“greencongruent”=congruent trial with green font\n“greenincongruent”=incongruent trial with green font\n“redcongruent”=congruent trial with red font\n“redincongruent”=incongruent trial with red font\n“yellowcongruent”=congruent trial with yellow font\n“yellowincongruent”=incongruent trial with yellow font
7,trialnum,The sequence number of the currently recorded trial
8,values.congruency,Provides information about the congruency of the current trial:
9,NaN,1 = congruent (color of word and meaning of word are the same);


In [3]:
#explore the data

print("Stroop info")
df_stroop.info()
display(df_stroop.head())

Stroop info
<class 'pandas.DataFrame'>
RangeIndex: 111300 entries, 0 to 111299
Data columns (total 14 columns):
 #   Column             Non-Null Count   Dtype
---  ------             --------------   -----
 0   build              111300 non-null  str  
 1   computer.platform  111300 non-null  str  
 2   date               111300 non-null  int64
 3   time               111300 non-null  str  
 4   blockcode          111300 non-null  str  
 5   blocknum           111300 non-null  int64
 6   trialcode          111300 non-null  str  
 7   trialnum           111300 non-null  int64
 8   values.congruency  111300 non-null  int64
 9   stimulusitem1      111300 non-null  str  
 10  response           111300 non-null  int64
 11  correct            111300 non-null  int64
 12  latency            111300 non-null  int64
 13  subjectid          111300 non-null  int64
dtypes: int64(8), str(6)
memory usage: 16.7 MB


,build,computer.platform,date,time,blockcode,blocknum,trialcode,trialnum,values.congruency,stimulusitem1,response,correct,latency,subjectid
0,5.0.6.0,win,111917,11:17:22,stroop,2,blackcongruent,2,1,czarny,37,1,1443,1
1,5.0.6.0,win,111917,11:17:22,stroop,2,redincongruent,3,2,zielony,33,0,1268,1
2,5.0.6.0,win,111917,11:17:22,stroop,2,greencongruent,4,1,zielony,33,1,1430,1
3,5.0.6.0,win,111917,11:17:22,stroop,2,redincongruent,5,2,niebieski,36,0,1098,1
4,5.0.6.0,win,111917,11:17:22,stroop,2,greencongruent,6,1,zielony,33,1,2061,1


In [4]:
#check for NaN

print("Tot NaN for Stroop-test")
df_stroop.isna().sum()

Tot NaN for Stroop-test


build                0
computer.platform    0
date                 0
time                 0
blockcode            0
blocknum             0
trialcode            0
trialnum             0
values.congruency    0
stimulusitem1        0
response             0
correct              0
latency              0
subjectid            0
dtype: int64

In [5]:
# läser in csv filen

# gör datumet mer lättläst
df_stroop['date'] = pd.to_datetime(df_stroop['date'].astype(str).str.zfill(6), format='%m%d%y', errors='coerce').dt.strftime('%Y-%m-%d')

In [ ]:
# sortera kronologiskt efter deltagare, datum och tid så sessionerna hamnar i rätt ordning
df_stroop = df_stroop.sort_values(by=['subjectid', 'date', 'time'])

In [ ]:
# skapa 'session'-kolumnen per deltagare (1 = tidigaste tillfället, 2 = nästa, 3 = sista)
df_stroop['session'] = df_stroop.groupby('subjectid')['date'].transform(lambda x: pd.factorize(x)[0] + 1)

In [ ]:
# välj enbart de kolumner vi vill behålla (inklusive 'session')
kolumner_att_behålla = ['date', 'time', 'subjectid', 'session', 'correct', 'latency', 'values.congruency']

In [ ]:
# kontrollera att kolumnerna finns
df_clean_stroop = df_stroop[kolumner_att_behålla].copy()

In [6]:
df_clean_stroop.head()

,date,time,subjectid,session,correct,latency,values.congruency
0,2017-11-19,11:17:22,1,1,1,1443,1
1,2017-11-19,11:17:22,1,1,0,1268,2
2,2017-11-19,11:17:22,1,1,1,1430,1
3,2017-11-19,11:17:22,1,1,0,1098,2
4,2017-11-19,11:17:22,1,1,1,2061,1


In [7]:
#antal unika deltagare
antal_unika = df_clean_stroop['subjectid'].nunique()
print(f"Antal unika deltagare: {antal_unika}")

Antal unika deltagare: 466


In [8]:
#create unique session by osrting for subject id and date
df_stroop['session'] = df_stroop.groupby('subjectid')['date'].transform(lambda x: pd.factorize(x)[0] + 1)

#control how it looks for person with several sessions
print(df_stroop[df_stroop['subjectid'] == 484][['subjectid', 'date', 'time', 'session']].drop_duplicates())

        subjectid        date      time  session
110880        484  2018-01-04  14:50:38        1
111020        484  2019-03-19  18:41:47        2
111160        484  2019-04-04  13:24:45        3
